In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, current_date

In [0]:
catalog_name = spark.conf.get("pipeline.catalog_name", "dbr_dev")
bronze_schema_name = spark.conf.get("pipeline.bronze_schema_name", "weather_bronze")
streaming_volume_path = f"/Volumes/{catalog_name}/{bronze_schema_name}/raw/streaming-weather/"

In [0]:
@dlt.table(
    name=f"{bronze_schema_name}.weather_streaming_data",
    comment="Bronze table for streaming weather data with Auto Loader.",
    table_properties={"quality": "bronze"}
)
def weather_streaming_data_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") #schema evolution
        .load(streaming_volume_path)
        .selectExpr("*", "_metadata.file_name as source_filename")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
    )